# JEPA-guided diffusion vs. standard denoisers: side-by-side outputs

Re-runs the trained adapter + frozen ADM 256×256 of `jepa-guided-diffusion` (v1) on the **same inputs and seeds** as its evaluation, next to four standard denoisers, and saves every output.

| method | what it is | told the noise level |
|---|---|---|
| ADM Tweedie / ADM reverse | frozen unconditional ADM: one-step posterior mean / full reverse chain from t* | yes (t*) |
| adapter null: reverse, adapter JEPA: Tweedie / reverse | ADM + the trained adapter, null or JEPA latents | yes (t*) |
| BM3D | classical collaborative filtering (`bm3d.bm3d_rgb`) | yes (σ) |
| DRUNet | CNN with a noise-level map (DPIR), trained on σ ≤ 50/255 | yes (σ) |
| Restormer | transformer, blind Gaussian model, trained on σ ≤ 50/255 | no |
| SCUNet | swin-conv UNet, blind, trained for real camera noise (`scunet_color_real_psnr`) | no |

Inputs: DIV2K valid 0801–0816 (centre 256 crop) at σ ∈ {0.1, 0.2, 0.3} (≈ 26, 51, 77 /255: σ = 0.3 is beyond the Gaussian models' training range) and the 4 PolyU real-noise pairs, for which the given σ is std(noisy − clean), as in the original notebook. Standard denoisers run on all 52 cases; the diffusion variants (≈ 0.75 s per reverse step on a T4) run on `DIFFUSION_IMAGES` and the real pairs. The inputs are checked against the original run's `scores.csv`.

Kaggle: GPU T4 ×2, internet on, outputs of `jepa-denoise-finetune` (v2) and `jepa-guided-diffusion` (v1) attached.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "bm3d", "einops", "timm"], check=True)

In [ ]:
import glob, importlib, importlib.util, io, json, math, os, random, re, subprocess, sys, time, types, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.checkpoint import checkpoint
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# ---- knobs ----
N_EVAL, SIGMAS = 16, [0.1, 0.2, 0.3]  # the jepa-guided-diffusion evaluation set: same images, seeds and order
DIFFUSION_IMAGES = [0, 1, 2, 3, 9]      # DIV2K cases that also run the slow diffusion variants (div2k_9: the JEPA failure case)
OVERVIEW_NOISE = "gaussian 0.2"

assert torch.cuda.is_available(), "GPU required"
D0 = torch.device("cuda:0")                                            # ADM, adapter, standard denoisers
D1 = torch.device("cuda:1") if torch.cuda.device_count() > 1 else D0  # I-JEPA
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], torch.__version__)
CACHE, OUT = "/tmp/compare_cache", "outputs"
for d in (CACHE, OUT, f"{OUT}/grids", f"{OUT}/images"):
    os.makedirs(d, exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
S = 256
TIMES = {}

found = glob.glob("/kaggle/input/**/jepa_adapter.pt", recursive=True)
assert found, "attach the output of the jepa-guided-diffusion notebook (version 1) as input data"
RUN_DIR = os.path.dirname(found[0])
ORIGINAL = pd.read_csv(f"{RUN_DIR}/scores.csv")  # the guided-diffusion run's per-case scores


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if size and have == size:
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def add_gaussian(x, sigma, seed=0):
    return (x + sigma * torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))).clamp(0, 1)


def load_zip_images(path, short):
    with zipfile.ZipFile(path) as z:
        names = sorted(n for n in z.namelist() if n.lower().endswith(".png"))

    def one(name):
        with zipfile.ZipFile(path) as z:
            im = Image.open(io.BytesIO(z.read(name))).convert("RGB")
        return torch.from_numpy(np.asarray(TF.resize(im, short)).copy()).permute(2, 0, 1).contiguous()

    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(one, names))


def fetch_image(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return Image.open(io.BytesIO(r.read())).convert("RGB")


def slug(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


def save_png(img, path):
    Image.fromarray((img.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)).save(path)

## 1 · Inputs: identical to the guided-diffusion evaluation
Same images, crops, noise seeds and case order (the reverse process of case *k* is seeded with *k*). Every noisy input is checked against the original run's noisy-input PSNR.

In [ ]:
t0 = time.time()
DIV2K = "https://data.vision.ee.ethz.ch/cvl/DIV2K/"
VALID_IMGS = load_zip_images(download(DIV2K + "DIV2K_valid_HR.zip", f"{CACHE}/DIV2K_valid_HR.zip", size=448_993_893), S)
EVAL_CLEAN = torch.stack([TF.center_crop(im, S).float() / 255 for im in VALID_IMGS[:N_EVAL]])
POLYU = "https://raw.githubusercontent.com/csjunxu/PolyU-Real-World-Noisy-Images-Dataset/master/CroppedImages/"
REAL = []
for n in ["Canon80D_8_8_3200_ball_1", "Canon5D2_5_160_6400_bicycle_10", "NikonD800_5_100_4000_flower_15", "Sony_4-5_125_3200_plant_13"]:
    noisy, clean = (TF.to_tensor(TF.center_crop(fetch_image(f"{POLYU}{n}_{k}.JPG"), S)) for k in ("real", "mean"))
    REAL.append((n, noisy, clean))

CASES = [dict(image=f"div2k_{i}", noise=f"gaussian {s}", sigma=s, clean=y, noisy=add_gaussian(y, s, seed=1000 * i + j))
         for i, y in enumerate(EVAL_CLEAN) for j, s in enumerate(SIGMAS)]
CASES += [dict(image=n, noise="real camera", sigma=float((noisy - clean).std()), clean=clean, noisy=noisy) for n, noisy, clean in REAL]
DIFFUSION_KEYS = {f"div2k_{i}" for i in DIFFUSION_IMAGES}
for k, case in enumerate(CASES):
    case["seed"] = k  # denoise_variants(..., seed=k) in the original notebook
    case["diffusion"] = case["noise"] == "real camera" or case["image"] in DIFFUSION_KEYS

noisy_psnr = ORIGINAL[ORIGINAL.method == "noisy input"].set_index(["image", "noise"]).psnr
INPUT_GAP = max(abs(peak_signal_noise_ratio(c["clean"].permute(1, 2, 0).numpy(), c["noisy"].permute(1, 2, 0).numpy(), data_range=1.0)
                    - noisy_psnr[(c["image"], c["noise"])]) for c in CASES)
print(f"{len(CASES)} cases, {sum(c['diffusion'] for c in CASES)} with diffusion variants; "
      f"max |noisy-input PSNR - original run| = {INPUT_GAP:.1e} dB")
assert INPUT_GAP < 1e-3, "the inputs differ from the guided-diffusion run"
TIMES["inputs"] = time.time() - t0

## 2 · Standard denoisers on all cases
Architecture files are fetched at pinned commits and loaded with `strict=True`; weights come from the authors' releases.

In [ ]:
import bm3d
import lpips

LPIPS = lpips.LPIPS(net="alex", verbose=False).to(D0)
SOURCES = {  # pinned commits: only the architecture files are needed
    "cszn/KAIR": ("fc1732f4a4514e42ce15e5b3a1e18c828af47a1e", ["models/basicblock.py", "models/network_unet.py"]),
    "cszn/SCUNet": ("52e440a80a655b01e0b41e9dd9bfe599bc11625e", ["models/network_scunet.py"]),
    "swz30/Restormer": ("68dc6ac472db26f16361150cb7a96a1bc87da93f", ["basicsr/models/archs/restormer_arch.py"]),
}
WEIGHTS = {
    "drunet_color.pth": ("https://github.com/cszn/KAIR/releases/download/v1.0/drunet_color.pth", 130_579_305),
    "scunet_color_real_psnr.pth": ("https://github.com/cszn/KAIR/releases/download/v1.0/scunet_color_real_psnr.pth", 71_982_841),
    "gaussian_color_denoising_blind.pth": ("https://github.com/swz30/Restormer/releases/download/v1.0/gaussian_color_denoising_blind.pth", 104_611_957),
}
for repo, (commit, paths) in SOURCES.items():
    for path in paths:
        dest = f"{CACHE}/{repo.split('/')[1]}/{path}"
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        download(f"https://raw.githubusercontent.com/{repo}/{commit}/{path}", dest)
W = {name: download(url, f"{CACHE}/{name}", size=size) for name, (url, size) in WEIGHTS.items()}


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


if importlib.util.find_spec("thop") is None:  # SCUNet imports thop.profile only for a FLOP count in its __main__ block
    sys.modules["thop"] = types.ModuleType("thop")
    sys.modules["thop"].profile = None
sys.path.insert(0, f"{CACHE}/KAIR")
from models.network_unet import UNetRes  # needs models.basicblock from the same checkout
sys.path.remove(f"{CACHE}/KAIR")
for name in [m for m in sys.modules if m == "models" or m.startswith("models.")]:
    del sys.modules[name]  # do not shadow any other top-level "models" package
SCUNetNet = load_module("network_scunet", f"{CACHE}/SCUNet/models/network_scunet.py").SCUNet
RestormerNet = load_module("restormer_arch", f"{CACHE}/Restormer/basicsr/models/archs/restormer_arch.py").Restormer

DRUNET = UNetRes(in_nc=4, out_nc=3, nc=[64, 128, 256, 512], nb=4, act_mode="R", downsample_mode="strideconv",
                 upsample_mode="convtranspose", bias=False)  # the released DPIR weights have no biases
DRUNET.load_state_dict(torch.load(W["drunet_color.pth"], map_location="cpu", weights_only=True), strict=True)
SCUNET = SCUNetNet(in_nc=3, config=[4, 4, 4, 4, 4, 4, 4], dim=64)
SCUNET.load_state_dict(torch.load(W["scunet_color_real_psnr.pth"], map_location="cpu", weights_only=True), strict=True)
RESTORMER = RestormerNet(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4,
                         heads=[1, 2, 4, 8], ffn_expansion_factor=2.66, bias=False, LayerNorm_type="BiasFree", dual_pixel_task=False)
RESTORMER.load_state_dict(torch.load(W["gaussian_color_denoising_blind.pth"], map_location="cpu", weights_only=True)["params"], strict=True)
for net in (DRUNET, SCUNET, RESTORMER):
    net.to(D0).eval().requires_grad_(False)
BASELINES = ["BM3D", "DRUNet", "Restormer", "SCUNet"]


def scores(out, clean):
    a, b = out.permute(1, 2, 0).numpy(), clean.permute(1, 2, 0).numpy()
    return dict(psnr=peak_signal_noise_ratio(b, a, data_range=1.0),
                ssim=structural_similarity(b, a, channel_axis=-1, data_range=1.0),
                lpips=LPIPS(out[None].to(D0) * 2 - 1, clean[None].to(D0) * 2 - 1).item())


@torch.no_grad()
def denoise_baselines(noisy, sigma):
    """Every standard denoiser on one noisy image [3,S,S] in [0,1]; BM3D and DRUNet are told sigma, the others are blind."""
    x = noisy[None].to(D0)
    outs = {"DRUNet": DRUNET(torch.cat([x, torch.full_like(x[:, :1], sigma)], 1)), "Restormer": RESTORMER(x), "SCUNet": SCUNET(x)}
    outs = {name: out[0].float().clamp(0, 1).cpu() for name, out in outs.items()}
    rgb = noisy.permute(1, 2, 0).numpy().astype(np.float64)
    outs["BM3D"] = torch.from_numpy(np.ascontiguousarray(bm3d.bm3d_rgb(rgb, sigma_psd=sigma))).permute(2, 0, 1).float().clamp(0, 1)
    return {name: outs[name] for name in BASELINES}


t0, rows, OUTPUTS = time.time(), [], {}
for k, case in enumerate(CASES):
    outs = denoise_baselines(case["noisy"], case["sigma"])
    rows += [dict(image=case["image"], noise=case["noise"], method=m, **scores(o, case["clean"])) for m, o in outs.items()]
    if case["diffusion"]:
        OUTPUTS[(case["image"], case["noise"])] = dict(outs)
    if k % 13 == 0:
        print(f"  case {k + 1}/{len(CASES)}  {(time.time() - t0) / 60:.1f} min", flush=True)
baseline_scores = pd.DataFrame(rows)
TIMES["standard denoisers"] = time.time() - t0
print(baseline_scores.groupby(["noise", "method"], sort=False)[["psnr", "ssim", "lpips"]].mean().round(3).to_string())
del DRUNET, SCUNET, RESTORMER
torch.cuda.empty_cache()

## 3 · JEPA-guided ADM: the trained adapter
The next two cells are copied verbatim from `jepa-guided-diffusion` (frozen ADM on GPU 0, denoising I-JEPA on GPU 1, adapter and conditional forward, with its equivalence asserts); then the trained adapter weights are loaded.

In [ ]:
GD, IJ = f"{CACHE}/guided-diffusion", f"{CACHE}/ijepa"
for url, path in (("https://github.com/openai/guided-diffusion", GD), ("https://github.com/facebookresearch/ijepa", IJ)):
    if not os.path.isdir(path):
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, path], check=True)
sys.path[:0] = [GD, IJ]
from guided_diffusion.script_util import create_model
from guided_diffusion.nn import timestep_embedding
import guided_diffusion.unet as gd_unet

# ADM's attention blocks always use its own checkpoint function, which differentiates w.r.t. the block weights and
# fails when they are frozen; eps_pred() checkpoints whole blocks with torch.utils.checkpoint instead.
gd_unet.checkpoint = lambda func, inputs, params, flag: func(*inputs)
from src.models import vision_transformer as vt

adm = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                   attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                   use_scale_shift_norm=True)  # the README's flags for 256x256_diffusion_uncond.pt
ADM_CKPT = download("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt",
                    f"{CACHE}/256x256_diffusion_uncond.pt", size=2_211_383_297)
adm.load_state_dict(torch.load(ADM_CKPT, map_location="cpu", weights_only=True))
adm.to(D0).eval().requires_grad_(False).convert_to_fp16()


def _sdpa_attention(self, x):  # same math as I-JEPA's attention, memory-efficient kernel
    B, n, C = x.shape
    q, k, v = self.qkv(x).reshape(B, n, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
    x = F.scaled_dot_product_attention(q, k, v, scale=self.scale)
    return self.proj(x.transpose(1, 2).reshape(B, n, C)), None


vt.Attention.forward = _sdpa_attention
found = glob.glob("/kaggle/input/**/jepa_denoise_vith14_fp16.pt", recursive=True)
assert found, "attach the output of the jepa-denoise-finetune notebook (version 2) as input data"
jepa = vt.vit_huge(patch_size=14)
jepa.load_state_dict(torch.load(found[0], map_location="cpu", weights_only=True)["target_encoder"])
jepa = jepa.half().to(D1).eval().requires_grad_(False)
JMEAN = torch.tensor([0.485, 0.456, 0.406], device=D1).view(1, 3, 1, 1)
JSTD = torch.tensor([0.229, 0.224, 0.225], device=D1).view(1, 3, 1, 1)


@torch.no_grad()
def jepa_latents(x):
    """x [B,3,S,S] in [0,1] -> [B,D,16,16]: LayerNormed target-encoder tokens of the 224 px resized image."""
    x = F.interpolate(x.to(D1).float(), size=(224, 224), mode="bilinear", antialias=True, align_corners=False)
    with torch.autocast("cuda", dtype=torch.float16):
        h = jepa((x - JMEAN) / JSTD)
    h = F.layer_norm(h.float(), (h.size(-1),))
    return h.transpose(1, 2).reshape(len(x), h.size(-1), 16, 16).to(D0)

In [ ]:
@torch.no_grad()
def injection_points(res_list=(32, 16, 8)):
    """{ADM encoder block index: (resolution, channels)} for the first block at each resolution."""
    emb = adm.time_embed(timestep_embedding(torch.zeros(1, dtype=torch.long, device=D0), adm.model_channels))
    h, points = torch.zeros(1, 3, S, S, device=D0).type(adm.dtype), {}
    for i, m in enumerate(adm.input_blocks):
        h = m(h, emb)
        if h.shape[-1] in res_list and h.shape[-1] not in [r for r, _ in points.values()]:
            points[i] = (h.shape[-1], h.shape[1])
    return points


def _zero(m):
    nn.init.zeros_(m.weight)
    nn.init.zeros_(m.bias)
    return m


class JEPAAdapter(nn.Module):
    """JEPA latents [B,D,16,16] -> features added to ADM encoder blocks + an offset on ADM's time embedding."""

    def __init__(self, d, points, emb_dim):
        super().__init__()
        self.res = {str(i): r for i, (r, _) in points.items()}
        self.branches = nn.ModuleDict({str(i): nn.Sequential(
            nn.Conv2d(d, ch, 1), nn.GroupNorm(32, ch), nn.SiLU(), nn.Conv2d(ch, ch, 3, padding=1), nn.SiLU(),
            _zero(nn.Conv2d(ch, ch, 1))) for i, (r, ch) in points.items()})
        self.glob = nn.Sequential(nn.Linear(d, emb_dim), nn.SiLU(), _zero(nn.Linear(emb_dim, emb_dim)))

    def forward(self, c):
        feats = {}
        for i, branch in self.branches.items():
            r = self.res[i]
            feats[int(i)] = branch(c if r == c.shape[-1] else F.interpolate(c, size=(r, r), mode="bilinear", align_corners=False))
        return feats, self.glob(c.mean((2, 3)))


def eps_pred(x, t, cond=None, grad_ckpt=False):
    """ADM's UNetModel.forward (ε channels only) with adapter features added after the injection blocks."""
    emb = adm.time_embed(timestep_embedding(t, adm.model_channels))
    feats = {}
    if cond is not None:
        feats, g = cond
        emb = emb + g
    run = (lambda m, *a: checkpoint(m, *a, use_reentrant=False)) if grad_ckpt else (lambda m, *a: m(*a))
    hs, h = [], x.type(adm.dtype)
    for i, m in enumerate(adm.input_blocks):
        h = run(m, h, emb)
        if i in feats:
            h = h + feats[i].type(h.dtype)
        hs.append(h)
    h = run(adm.middle_block, h, emb)
    for m in adm.output_blocks:
        h = run(m, torch.cat([h, hs.pop()], 1), emb)
    return adm.out(h.type(x.dtype))[:, :3].float()


INJECT = injection_points()
adapter = JEPAAdapter(jepa.embed_dim, INJECT, adm.model_channels * 4).to(D0)
print("injection points {block: (px, channels)}:", INJECT, f"| adapter {sum(p.numel() for p in adapter.parameters()) / 1e6:.1f}M params")

BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)  # ADM's linear schedule
ABAR = torch.cumprod(1 - BETAS, 0)
ABAR_D = ABAR.float().to(D0)

with torch.no_grad():  # the re-implemented forward equals ADM's, and the untrained adapter changes nothing
    xt, tt = torch.randn(2, 3, S, S, device=D0), torch.tensor([10, 500], device=D0)
    ref = adm(xt, tt)[:, :3].float()
    assert torch.allclose(eps_pred(xt, tt), ref, atol=1e-3)
    assert torch.allclose(eps_pred(xt, tt, adapter(jepa_latents(EVAL_CLEAN[:2]))), ref, atol=1e-3)

In [ ]:
adapter.load_state_dict(torch.load(f"{RUN_DIR}/jepa_adapter.pt", map_location="cpu", weights_only=True))
adapter.eval()


def t_for_sigma(sigma):  # pixel σ on [0, 1] -> step whose noise level matches (x in [-1, 1] doubles σ)
    return int((ABAR - 1 / (1 + (2 * sigma) ** 2)).abs().argmin())


def to01(z):
    return ((z + 1) / 2).clamp(0, 1).cpu()


@torch.no_grad()
def denoise_variants(y, sigma, c, seed=0):
    """Reverse process from t* for 3 variants in one batch: ADM, adapter(null), adapter(JEPA latents c [1,D,16,16]).
    Returns (Tweedie estimates [3,3,S,S], posterior samples [3,3,S,S])."""
    adapter.eval()
    fj, gj = adapter(c)
    fn, gn = adapter(torch.zeros_like(c))
    cond = ({i: torch.cat([torch.zeros_like(fj[i]), fn[i], fj[i]]) for i in fj}, torch.cat([torch.zeros_like(gj), gn, gj]))
    t0 = t_for_sigma(sigma)
    gen = torch.Generator(device=D0).manual_seed(seed)
    x = (ABAR[t0].sqrt().float() * (2 * y.to(D0) - 1))[None].repeat(3, 1, 1, 1)
    tweedie = None
    for t in range(t0, -1, -1):
        a_t, beta_t = ABAR[t].item(), BETAS[t].item()
        a_prev = ABAR[t - 1].item() if t > 0 else 1.0
        eps = eps_pred(x, torch.full((3,), t, device=D0, dtype=torch.long), cond)
        x0 = ((x - (1 - a_t) ** 0.5 * eps) / a_t ** 0.5).clamp(-1, 1)
        if tweedie is None:
            tweedie = x0
        if t == 0:
            x = x0
            break
        mean = (a_prev ** 0.5 * beta_t / (1 - a_t)) * x0 + ((1 - beta_t) ** 0.5 * (1 - a_prev) / (1 - a_t)) * x
        noise = torch.randn((1, 3, S, S), generator=gen, device=D0).expand_as(x)  # same draw for every variant
        x = mean + (beta_t * (1 - a_prev) / (1 - a_t)) ** 0.5 * noise
    return to01(tweedie), to01(x)


DIFF_METHODS = ["ADM Tweedie", "ADM reverse", "adapter null: reverse", "adapter JEPA: Tweedie", "adapter JEPA: reverse"]
t0, rows = time.time(), []
for case in [c for c in CASES if c["diffusion"]]:
    tw, smp = denoise_variants(case["noisy"], case["sigma"], jepa_latents(case["noisy"][None]), seed=case["seed"])
    outs = dict(zip(DIFF_METHODS, [tw[0], smp[0], smp[1], tw[2], smp[2]]))
    rows += [dict(image=case["image"], noise=case["noise"], method=m, **scores(o, case["clean"])) for m, o in outs.items()]
    OUTPUTS[(case["image"], case["noise"])].update(outs)
    print(f"  {case['image']} | {case['noise']}: t* = {t_for_sigma(case['sigma'])}  {(time.time() - t0) / 60:.1f} min", flush=True)
diffusion_scores = pd.DataFrame(rows)
TIMES["diffusion variants"] = time.time() - t0

# Same inputs and seeds as the guided-diffusion run: how close are the re-run scores to its scores.csv?
merged = diffusion_scores.merge(ORIGINAL, on=["image", "noise", "method"], suffixes=("", "_run"))
RERUN_GAP = {m: float((g.psnr - g.psnr_run).abs().max()) for m, g in merged.groupby("method")}
print("max |PSNR re-run - original run| per method (dB):", {m: round(v, 4) for m, v in RERUN_GAP.items()})

## 4 · Tables and figures
`all_scores.csv` combines the original run's diffusion scores (all 52 cases; inputs verified identical above) with the standard denoisers. Grids and PNGs show this notebook's outputs for the diffusion cases.

In [ ]:
METHODS = ["noisy input", *DIFF_METHODS, *BASELINES]
all_scores = pd.concat([ORIGINAL[ORIGINAL.method.isin(METHODS)], baseline_scores], ignore_index=True)
all_scores.to_csv(f"{OUT}/all_scores.csv", index=False)
baseline_scores.to_csv(f"{OUT}/standard_denoisers_scores.csv", index=False)
diffusion_scores.to_csv(f"{OUT}/diffusion_rerun_scores.csv", index=False)
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()} over all cases (" + ("lower" if metric == "lpips" else "higher") + " = better)")
    print(all_scores.pivot_table(index="method", columns="noise", values=metric).reindex(METHODS).round(3).to_string())

SHOWN = pd.concat([ORIGINAL[ORIGINAL.method == "noisy input"], diffusion_scores, baseline_scores]).set_index(["image", "noise", "method"])


def label(key, method):
    r = SHOWN.loc[(*key, method)]
    return f"{method}\n{r.psnr:.2f} dB | LPIPS {r.lpips:.3f}"


def show(ax, img, title, zoom=False):
    ax.imshow((img[:, 64:192, 64:192] if zoom else img).permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])


for case in [c for c in CASES if c["diffusion"]]:
    key = (case["image"], case["noise"])
    panels = [("clean (reference)", case["clean"]), ("noisy input", case["noisy"])] + [(m, OUTPUTS[key][m]) for m in METHODS[1:]]
    fig, axs = plt.subplots(2, len(panels), figsize=(2.2 * len(panels), 5.0))
    for j, (name, img) in enumerate(panels):
        show(axs[0, j], img, name if j == 0 else label(key, name))
        show(axs[1, j], img, "zoom", zoom=True)
    fig.suptitle(f"{case['image']} | {case['noise']} (σ = {case['sigma']:.3f})", fontsize=10)
    plt.tight_layout(); plt.savefig(f"{OUT}/grids/{slug(case['image'])}__{slug(case['noise'])}.png", dpi=100); plt.close(fig)
    folder = f"{OUT}/images/{slug(case['image'])}__{slug(case['noise'])}"
    os.makedirs(folder, exist_ok=True)
    for j, (name, img) in enumerate(panels):
        save_png(img, f"{folder}/{j:02d}-{slug(name)}.png")

OVERVIEW = ["noisy input", "ADM Tweedie", "ADM reverse", "adapter JEPA: reverse", *BASELINES]
picked = [c for c in CASES if c["diffusion"] and c["noise"] == OVERVIEW_NOISE] + [c for c in CASES if c["noise"] == "real camera"][:1]
fig, axs = plt.subplots(len(picked), len(OVERVIEW) + 1, figsize=(2.1 * (len(OVERVIEW) + 1), 2.3 * len(picked)), squeeze=False)
for r, case in enumerate(picked):
    key = (case["image"], case["noise"])
    for j, (name, img) in enumerate([("clean", case["clean"])] + [(m, case["noisy"] if m == "noisy input" else OUTPUTS[key][m]) for m in OVERVIEW]):
        show(axs[r, j], img, name if j == 0 else label(key, name))
    axs[r, 0].set_ylabel(f"{case['image'][:14]}\n{case['noise']}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/overview.png", dpi=100); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(16, 4.8))
for a, metric in zip(axs, ("psnr", "lpips")):
    all_scores.groupby(["method", "noise"])[metric].mean().unstack().reindex(METHODS).plot.bar(ax=a, rot=35, width=0.8)
    a.set_title(f"{metric.upper()}, mean over cases (" + ("lower" if metric == "lpips" else "higher") + " = better)")
    a.set_xlabel("")
    a.legend(fontsize=8, title="noise")
plt.tight_layout(); plt.savefig(f"{OUT}/metrics.png", dpi=110); plt.show()

with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(sources={repo: commit for repo, (commit, _) in SOURCES.items()}, weights={k: v[0] for k, v in WEIGHTS.items()},
                   diffusion_images=DIFFUSION_IMAGES, input_psnr_gap_db=INPUT_GAP, rerun_psnr_gap_db=RERUN_GAP,
                   minutes={k: v / 60 for k, v in TIMES.items()},
                   mean=all_scores.groupby(["noise", "method"], sort=False)[["psnr", "ssim", "lpips"]].mean().reset_index().to_dict("records")),
              f, indent=1, default=float)
print(sorted(os.listdir(OUT)))